In [11]:
%%writefile app.py
import streamlit as st
import numpy as np
import joblib

model = joblib.load("insurance_model.pkl")

st.title("Medical Insurance Cost Prediction")

age = st.slider("Age",18,65)

sex = st.selectbox("Sex",["Male","Female"])

bmi = st.number_input("BMI",10.0,50.0)

children = st.slider("Children",0,5)

smoker = st.selectbox("Smoker",["Yes","No"])

region = st.selectbox("Region",["southwest","southeast","northwest","northeast"])

sex = 1 if sex=="Male" else 0
smoker = 1 if smoker=="Yes" else 0

region_dict={
"southwest":0,
"southeast":1,
"northwest":2,
"northeast":3
}

region=region_dict[region]

if st.button("Predict Insurance Cost"):
    data=np.array([[age,sex,bmi,children,smoker,region]])
    prediction=model.predict(data)
    st.success(f"Estimated Insurance Cost: ${prediction[0]:.2f}")

Overwriting app.py


In [12]:
!ls

app.py	cloudflared  insurance.csv  insurance_model.pkl  sample_data


In [13]:
from google.colab import files
uploaded = files.upload()

Saving insurance.csv to insurance (1).csv


In [ ]:
import pandas as pd

df = pd.read_csv("insurance.csv")

df.head()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df['sex'] = le.fit_transform(df['sex'])
df['smoker'] = le.fit_transform(df['smoker'])
df['region'] = le.fit_transform(df['region'])

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop("charges", axis=1)
y = df["charges"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor()

rf.fit(X_train, y_train)

RandomForestRegressor()

In [ ]:
import joblib

joblib.dump(rf, "insurance_model.pkl")

['insurance_model.pkl']

In [ ]:
!streamlit run app.py &




  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.87.36.0:8501

  Stopping...


In [ ]:
!chmod +x cloudflared

In [ ]:
!pip install streamlit pyngrok

In [ ]:
!pip install gradio

In [10]:
import gradio as gr
import numpy as np
import joblib

# Load trained model
model = joblib.load("insurance_model.pkl")

def predict(age, sex, bmi, children, smoker, region):

    # Encode categorical inputs
    sex = 1 if sex == "Male" else 0
    smoker = 1 if smoker == "Yes" else 0

    region_dict = {
        "southwest": 0,
        "southeast": 1,
        "northwest": 2,
        "northeast": 3
    }

    region = region_dict[region]

    # Prepare input data
    data = np.array([[age, sex, bmi, children, smoker, region]])

    # Model prediction
    prediction = model.predict(data)

    # Convert USD to INR
    inr = prediction[0] * 83

    return f"Estimated Insurance Cost: ₹{inr:,.2f}"


# Gradio Interface
demo = gr.Interface(
    fn=predict,
    inputs=[
        gr.Slider(18, 65, label="Age"),
        gr.Dropdown(["Male", "Female"], label="Sex"),
        gr.Number(label="BMI"),
        gr.Slider(0, 5, label="Children"),
        gr.Dropdown(["Yes", "No"], label="Smoker"),
        gr.Dropdown(["southwest", "southeast", "northwest", "northeast"], label="Region")
    ],
    outputs="text",
    title="Medical Insurance Cost Prediction",
    description="Predict medical insurance charges using Machine Learning"
)

# Launch app
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a316557f549daa90cb.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
